## 1. Import necessary libraries

In [3]:
import cv2
import numpy as np
import requests

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Requests version:", requests.__version__)

print("All libraries imported successfully!")

OpenCV version: 5.0.0
NumPy version: 2.5.4
Requests version: 2.34.2
All libraries imported successfully!


## 2. Camera configuration and connection test

In [8]:
import cv2
import json

# Load camera configuration
with open("../config/camera_config.json", "r") as file:
    camera_config = json.load(file)

camera_name = camera_config["camera_name"]
stream_url = camera_config["stream_url"]

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if camera.isOpened():
    print(f"{camera_name} connected successfully!")

    # Read one frame
    success, frame = camera.read()

    if success:
        height, width, channels = frame.shape
        print("Frame captured successfully!")
        print(f"Resolution: {width} x {height}")
        print(f"Frame shape: {frame.shape}")
    else:
        print("Could not read a frame.")
else:
    print("Could not connect to the camera.")

# Release the camera
camera.release()

Phone Camera connected successfully!
Frame captured successfully!
Resolution: 1920 x 1080
Frame shape: (1080, 1920, 3)


## 3. Live Video Streaming

Here, OpenCV is used to continuously capture and display video frames from the connected IP camera. The VideoCapture() function establishes the connection, while read() retrieves each frame. The imshow() function displays the live video and waitKey() checks for keyboard input. Pressing the Esc key stops the stream and releases the camera.

In [9]:
# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Live streaming started. Press ESC to exit.")

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Display the live video
            cv2.imshow(camera_name, frame)

            # Check if ESC was pressed
            key = cv2.waitKey(1) & 0xFF

            if key == 27:
                print("Streaming stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Live streaming started. Press ESC to exit.
Streaming stopped.
Camera resources released.


## 4. Frame Processing and Information Overlays

In this section, each video frame is processed before being displayed. The frame is resized to a fixed width while maintaining its original aspect ratio. Additional information is displayed on the video, including the camera name, current date and time, resolution and estimated frames per second (FPS). FPS measures how many frames are processed and displayed per second.

In [10]:
import time
from datetime import datetime

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Frame processing started. Press ESC to exit.")

    previous_time = time.time()

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Get original frame dimensions
            height, width, channels = frame.shape

            # Resize while preserving aspect ratio
            display_width = camera_config["display_width"]
            display_height = int(height * display_width / width)

            frame = cv2.resize(frame, (display_width, display_height))

            # Calculate estimated FPS
            current_time = time.time()
            time_difference = current_time - previous_time

            fps = 1 / time_difference if time_difference > 0 else 0
            previous_time = current_time

            # Get current date and time
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Get resized resolution
            resolution = f"{display_width} x {display_height}"

            # Add information overlays
            cv2.putText(frame, camera_name, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(frame, timestamp, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"Resolution: {resolution}", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Display processed video
            cv2.imshow("IP Camera - Processed Stream", frame)

            # Exit when ESC is pressed
            key = cv2.waitKey(1) & 0xFF

            if key == 27:
                print("Streaming stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Frame processing started. Press ESC to exit.
Streaming stopped.
Camera resources released.


## 5. Screenshot Capture

This introduces screenshot capture using keyboard input. When the user presses the S key, the current processed video frame is saved as an image in the screenshots directory. Each screenshot receives a timestamped filename to prevent existing images from being overwritten. A confirmation message is displayed after a successful capture.

In [11]:
import os
import time
from datetime import datetime

# Create screenshots directory if it does not exist
os.makedirs("../screenshots", exist_ok=True)

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Screenshot capture ready. Press S to capture, ESC to exit.")

    previous_time = time.time()
    confirmation_until = 0

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Resize frame while preserving aspect ratio
            height, width, channels = frame.shape
            display_width = camera_config["display_width"]
            display_height = int(height * display_width / width)

            frame = cv2.resize(frame, (display_width, display_height))

            # Calculate FPS
            current_time = time.time()
            time_difference = current_time - previous_time
            fps = 1 / time_difference if time_difference > 0 else 0
            previous_time = current_time

            # Prepare information overlays
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            resolution = f"{display_width} x {display_height}"

            cv2.putText(frame, camera_name, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(frame, timestamp, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"Resolution: {resolution}", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Display screenshot confirmation
            if time.time() < confirmation_until:
                cv2.putText(frame, "Screenshot saved!", (10, 160),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # Display video
            cv2.imshow("IP Camera - Screenshot Capture", frame)

            key = cv2.waitKey(1) & 0xFF

            # Capture screenshot when S is pressed
            if key == ord("s") or key == ord("S"):
                filename = datetime.now().strftime("screenshot_%Y%m%d_%H%M%S_%f.jpg")
                filepath = os.path.join("../screenshots", filename)

                if cv2.imwrite(filepath, frame):
                    print(f"Screenshot saved: {filename}")
                    confirmation_until = time.time() + 2

            # Exit when ESC is pressed
            if key == 27:
                print("Screenshot capture stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Screenshot capture ready. Press S to capture, ESC to exit.
Screenshot saved: screenshot_20261114_190255_465677.jpg
Screenshot saved: screenshot_20261114_190402_595198.jpg
Screenshot capture stopped.
Camera resources released.


## 6. Video Recording

Here, we implement video recording using OpenCV's VideoWriter class. The user can press the R key to start recording and press it again to stop. Recorded videos are saved in the recordings directory using unique timestamped filenames. While recording, the application displays a recording indicator and the elapsed recording time. The video writer is released when recording stops or when the application exits.

In [12]:
import os
import time
from datetime import datetime

# Recording settings
os.makedirs("../recordings", exist_ok=True)

recording_fps = 20.0
video_writer = None
recording_start_time = None

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Video recording ready. Press R to start/stop, ESC to exit.")

    previous_time = time.time()

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Resize frame while preserving aspect ratio
            height, width, channels = frame.shape
            display_width = camera_config["display_width"]
            display_height = int(height * display_width / width)

            frame = cv2.resize(frame, (display_width, display_height))

            # Calculate FPS
            current_time = time.time()
            time_difference = current_time - previous_time
            fps = 1 / time_difference if time_difference > 0 else 0
            previous_time = current_time

            # Prepare information overlays
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            resolution = f"{display_width} x {display_height}"

            cv2.putText(frame, camera_name, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(frame, timestamp, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"Resolution: {resolution}", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Display recording status and elapsed time
            if video_writer is not None:
                elapsed_time = int(time.time() - recording_start_time)
                minutes, seconds = divmod(elapsed_time, 60)

                cv2.putText(frame, "RECORDING", (10, 160),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                cv2.putText(frame, f"Elapsed: {minutes:02d}:{seconds:02d}", (10, 190),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            # Display video
            cv2.imshow("IP Camera - Video Recording", frame)

            # Save frames while recording
            if video_writer is not None:
                video_writer.write(frame)

            key = cv2.waitKey(1) & 0xFF

            # Start or stop recording using R
            if key == ord("r") or key == ord("R"):

                if video_writer is None:
                    filename = datetime.now().strftime("recording_%Y%m%d_%H%M%S_%f.mp4")
                    filepath = os.path.join("../recordings", filename)

                    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

                    video_writer = cv2.VideoWriter(
                        filepath,
                        fourcc,
                        recording_fps,
                        (display_width, display_height)
                    )

                    if video_writer.isOpened():
                        recording_start_time = time.time()
                        print(f"Recording started: {filename}")
                    else:
                        print("Could not start recording.")
                        video_writer.release()
                        video_writer = None

                else:
                    video_writer.release()
                    video_writer = None
                    recording_start_time = None
                    print("Recording stopped and saved.")

            # Exit when ESC is pressed
            if key == 27:
                print("Streaming stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        # Release the video writer if recording is active
        if video_writer is not None:
            video_writer.release()
            print("Active recording saved.")

        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Video recording ready. Press R to start/stop, ESC to exit.
Recording started: recording_20261114_191144_681911.mp4
Recording stopped and saved.
Streaming stopped.
Camera resources released.


## 7. Error Handling and Camera Reconnection

This improves the reliability of the camera streaming system. It handles invalid camera addresses, unavailable cameras and failed frame reads. When the connection is interrupted, the application attempts to reconnect a limited number of times before stopping. The application also uses exception handling and a finally block to release the camera and video writer safely. This helps prevent resources from remaining occupied after the application exits.

In [23]:
import cv2
import os
import time
from datetime import datetime

# Reconnection settings
MAX_RECONNECT_ATTEMPTS = 5
RECONNECT_DELAY = 2
MAX_FAILED_READS = 5

# Recording settings
os.makedirs("../recordings", exist_ok=True)

recording_fps = 20.0
video_writer = None
recording_start_time = None

# Function to connect to the camera
def connect_camera():
    camera = cv2.VideoCapture(stream_url)

    if camera.isOpened():
        return camera

    camera.release()
    return None


# Initial camera connection
camera = connect_camera()

if camera is None:
    print("Camera unavailable. Attempting to connect...")

    for attempt in range(1, MAX_RECONNECT_ATTEMPTS + 1):
        print(f"Connection attempt {attempt}/{MAX_RECONNECT_ATTEMPTS}")
        time.sleep(RECONNECT_DELAY)

        camera = connect_camera()

        if camera is not None:
            print("Camera connected successfully!")
            break

if camera is None:
    print("Could not connect to the camera. Check the URL and network.")

else:
    print("Camera streaming started.")
    print("Press R to start/stop recording or ESC to exit.")

    failed_reads = 0
    previous_time = time.time()

    try:
        while True:
            success, frame = camera.read()

            # Handle failed frame reads
            if not success:
                failed_reads += 1
                print(f"Frame read failed ({failed_reads}/{MAX_FAILED_READS})")

                if failed_reads < MAX_FAILED_READS:
                    time.sleep(0.2)
                    continue

                print("Camera connection lost.")
                camera.release()
                camera = None

                # Attempt to reconnect
                for attempt in range(1, MAX_RECONNECT_ATTEMPTS + 1):
                    print(f"Reconnection attempt {attempt}/{MAX_RECONNECT_ATTEMPTS}")
                    time.sleep(RECONNECT_DELAY)

                    camera = connect_camera()

                    if camera is not None:
                        print("Camera reconnected successfully!")
                        failed_reads = 0
                        previous_time = time.time()
                        break

                if camera is None:
                    print("Reconnection failed. Stopping stream.")
                    break

                continue

            failed_reads = 0

            # Resize frame while preserving aspect ratio
            height, width, channels = frame.shape
            display_width = camera_config["display_width"]
            display_height = int(height * display_width / width)

            frame = cv2.resize(frame, (display_width, display_height))

            # Calculate FPS
            current_time = time.time()
            time_difference = current_time - previous_time
            fps = 1 / time_difference if time_difference > 0 else 0
            previous_time = current_time

            # Add information overlays
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            cv2.putText(frame, camera_name, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(frame, timestamp, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"Resolution: {display_width} x {display_height}", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Display recording status
            if video_writer is not None:
                elapsed_time = int(time.time() - recording_start_time)
                minutes, seconds = divmod(elapsed_time, 60)

                cv2.putText(frame, "RECORDING", (10, 160),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                cv2.putText(frame, f"Elapsed: {minutes:02d}:{seconds:02d}", (10, 190),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                video_writer.write(frame)

            # Display the video
            cv2.imshow("IP Camera - Error Handling", frame)

            key = cv2.waitKey(1) & 0xFF

            # Start or stop recording
            if key == ord("r") or key == ord("R"):

                if video_writer is None:
                    filename = datetime.now().strftime("recording_%Y%m%d_%H%M%S_%f.mp4")
                    filepath = os.path.join("../recordings", filename)

                    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

                    video_writer = cv2.VideoWriter(
                        filepath,
                        fourcc,
                        recording_fps,
                        (display_width, display_height)
                    )

                    if video_writer.isOpened():
                        recording_start_time = time.time()
                        print(f"Recording started: {filename}")
                    else:
                        video_writer.release()
                        video_writer = None
                        print("Could not start recording.")

                else:
                    video_writer.release()
                    video_writer = None
                    recording_start_time = None
                    print("Recording stopped and saved.")

            # Exit when ESC is pressed
            if key == 27:
                print("ESC pressed. Stopping stream.")
                break

    except KeyboardInterrupt:
        print("Application interrupted by user.")

    except Exception as error:
        print(f"An unexpected error occurred: {error}")

    finally:
        # Release active recording
        if video_writer is not None:
            video_writer.release()
            print("Recording saved and video writer released.")

        # Release camera
        if camera is not None:
            camera.release()

        cv2.destroyAllWindows()
        print("All camera resources released.")

Camera streaming started.
Press R to start/stop recording or ESC to exit.
Recording started: recording_20261114_193626_942170.mp4
Application interrupted by user.
Recording saved and video writer released.
All camera resources released.


In [17]:
stream_url = "http://192.168.3.101:8080/video"

## 8. Bonus Challenge - Motion Detection

This implements real-time motion detection using OpenCV's Background Subtraction algorithm (MOG2). The application identifies moving objects, draws bounding boxes
around detected movement and displays a motion detection indicator.

In [24]:
import cv2

# Create the background subtractor
background_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

# Connect to the camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Motion detection started. Press ESC to exit.")

    while True:
        success, frame = camera.read()

        if not success:
            print("Could not read camera frame.")
            break

        # Resize frame
        height, width = frame.shape[:2]
        new_height = int(height * 960 / width)
        frame = cv2.resize(frame, (960, new_height))

        # Apply background subtraction
        foreground_mask = background_subtractor.apply(frame)

        # Remove shadows from the mask
        _, foreground_mask = cv2.threshold(
            foreground_mask, 200, 255, cv2.THRESH_BINARY
        )

        # Find moving objects
        contours, _ = cv2.findContours(
            foreground_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        motion_detected = False

        for contour in contours:
            area = cv2.contourArea(contour)

            # Ignore very small movements
            if area > 1000:
                x, y, w, h = cv2.boundingRect(contour)

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                motion_detected = True

        # Display motion status
        if motion_detected:
            cv2.putText(
                frame,
                "MOTION DETECTED",
                (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )
        else:
            cv2.putText(
                frame,
                "No Motion",
                (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        cv2.imshow("IP Camera Motion Detection", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == 27:
            print("Motion detection stopped.")
            break

    camera.release()
    cv2.destroyAllWindows()

Motion detection started. Press ESC to exit.
Motion detection stopped.
